# Birch and Swinnerton-Dyer, counting points mod p

*maths · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/maths/birch_swinnerton_dyer)

Count the points of five elliptic curves of rank 0 to 4 over $\mathbb F_p$ for every prime up to $X$ at once, watch the products $\prod_{p\le X} N_p/p$ grow like $(\log X)^r$ as Birch and Swinnerton-Dyer first observed, and compute $L(E, s)$ at $s = 1$, where it vanishes to an order equal to the rank. The conjecture that this always happens is open; it is proven only for analytic rank 0 and 1.

**Vector form**

$$a_p = p + 1 - \#E(\mathbb F_p) = -\sum_{x \in \mathbb F_p} \Big(\frac{4x^3 + b_2x^2 + 2b_4x + b_6}{p}\Big),\qquad \prod_{p \le X} \frac{N_p}{p} \approx C\,(\log X)^{r},\qquad \operatorname{ord}_{s=1} L(E, s) = r = \operatorname{rank} E(\mathbb Q)$$

**Index (tensor) form**

$$a_{cp} = -\sum_x \chi_{p}\big(f_c(x) \bmod p\big),\qquad L^{(r)}_c(1) = \frac{4\pi}{\sqrt{N_c}} \sum_m \sum_n a_{cn}\, e^{-2\pi n e^{u_m}/\sqrt{N_c}}\, u_m^{r}\, e^{u_m} w_m \quad (\varepsilon_c = (-1)^r)$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $f_c(x) \bmod p$ | `(((4 * x + b2) % p * x + 2 * b4) % p * x + b6) % p` | (C, P_b, X) | every curve, every prime of a block and every $x \in \mathbb F_p$ at once (Horner, reduced mod $p$ at each step) |
| $\chi_p(v)$ | `is_square.scatter_(1, x * x % p, True)` | (P_b, X) bool | Legendre symbol from a table of squares $y^2 \bmod p$; $\chi = 0$ for $v = 0$ |
| $a_{cp}$ | `-chi.sum(dim=-1)` | (C, P) | the Frobenius trace; Hasse: $/a_p/ \le 2\sqrt p$ |
| $\log\prod_{p\le X} N_p/p$ | `torch.cumsum(log((p + 1 - a_p) / p), dim=1)` | (C, P) | the partial products for every $X$ at once (good primes only) |
| $r_{\mathrm{est}}$ | `least-squares slope against log(log(p))` | (C,) | slope over $\sqrt{X} \le p \le X$; converges very slowly |
| $a_{cn}$ | `new[:, q::q] += a_q[:, None] * a[:, 1 : n // q + 1]` | (C, n_{\max}) | Dirichlet coefficients by multiplicativity, one strided slice per prime power $q = p^k$ |
| $\sum_n a_{cn} e^{-2\pi n e^{u_m}/\sqrt{N}}$ | `einsum('n,nu->u', a, kernel)` | (U,) | repeated $n$ ⇒ summed: $F(t)$ at the Gauss–Legendre nodes $t = e^{u_m}$ |
| $L^{(r)}(1)$ | `2 * scale * einsum('ru,u->r', u**r, weight)` | (R,) | all derivatives at $s = 1$ at once; the first nonzero one gives the analytic rank |

An elliptic curve $E: y^2 + a_1xy + a_3y = x^3 + a_2x^2 + a_4x + a_6$ over $\mathbb Q$ has a finitely generated group of rational points $E(\mathbb Q) \cong \mathbb Z^r \oplus T$ (Mordell 1922); the rank $r$ is hard to compute. In the early 1960s Birch and Swinnerton-Dyer counted points modulo primes on the EDSAC computer and noticed that $\prod_{p \le X} N_p/p$ grows like $(\log X)^r$: more rational points force more points mod $p$. They conjectured that $L(E, s)$, built from the $a_p$, vanishes at $s = 1$ to order exactly $r$, with leading coefficient $\frac{L^{(r)}(E,1)}{r!} = \frac{\Omega\, R\, \#\text{Ш}\, \prod_p c_p}{\#T^2}$. It is a Clay Millennium Prize problem and open. Proven: if the analytic rank is 0 or 1, the rank equals it and Ш is finite (Gross and Zagier 1986, Kolyvagin 1989), using modularity (Wiles, Taylor–Wiles 1995, Breuil–Conrad–Diamond–Taylor 2001), which is what makes $L(E, s)$ continue to $s = 1$ at all. For analytic rank $\ge 2$ nothing general is known. The five curves are the smallest-conductor curves of ranks 0 to 4 (11a1, 37a1, 389a1, 5077a1 and 234446a1 in Cremona's tables). Here: the traces obey Hasse's bound $|a_p| \le 2\sqrt p$; for 11a1 they equal the coefficients of the modular form $\eta(q)^2\eta(q^{11})^2$ (tested); the root numbers are $\varepsilon = (-1)^r$; the first nonvanishing derivative at $s = 1$ is the $r$-th for every curve, and $L(E,1) = 0.2538418608\ldots$, $L'(E,1) = 0.3059997738\ldots$, $L''(E,1)/2 = 0.7593165\ldots$ and $L'''(E,1)/6 = 1.7318499\ldots$ match the LMFDB. For 11a1 the full formula checks out: $L(E,1)/\Omega = 1/5 = \#\text{Ш}\cdot c_{11}/\#T^2 = 1 \cdot 5 / 5^2$. A numerical zero of a derivative is evidence, not a proof that it vanishes. The partial products are the original heuristic: Goldfeld (1982) showed that if they really behave like $C(\log X)^r$ then $r$ is the analytic rank and the Riemann hypothesis holds for $L(E, s)$, so their convergence is itself conjectural and slow. The slopes order ranks 0 to 3 correctly by $X = 2\times 10^4$, but the rank-4 curve still lags behind its rank at $X = 3\times 10^4$. The Sato–Tate distribution of $a_p/2\sqrt p$, the semicircle $\frac{2}{\pi}\sqrt{1-x^2}$, is a theorem for curves without complex multiplication (Taylor and collaborators, 2008–2011).


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/maths/birch_swinnerton_dyer/main.py
"""Main loop: curves → a_p for every prime up to X → partial products, Sato–Tate, L(E, s) → plot."""

import math
from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    x_max = 3000 if fast_mode() else params["x_max"]
    coeffs = tensors.curve_coefficients(dev)  # (C, 5)
    names = list(tensors.CURVES)

    # 1. point counts over F_p for every prime p ≤ X and every curve at once
    primes = tensors.primes_up_to(x_max, dev)  # (P,)
    a_p = equation.frobenius_traces(coeffs, primes)  # (C, P)
    hasse = (a_p.abs().double() / (2 * primes.double().sqrt())).max().item()
    print(f"{len(names)} curves × {len(primes):,} primes ≤ {x_max:,}: max |a_p| / 2√p = {hasse:.4f} (Hasse: ≤ 1)")
    print("a_p for p = 2, 3, 5, 7, 11, 13:")
    for name, row in zip(names, a_p[:, :6].tolist()):
        print(f"  {name:>9}: {row}")

    # 2. Birch and Swinnerton-Dyer's experiment: ∏_{p ≤ X} N_p / p ≈ C (log X)^r
    log_prod = solve.log_partial_products(coeffs, primes)  # (C, P)
    slope = solve.rank_estimate(log_prod, primes)

    # 3. the L-function at s = 1: root number, order of vanishing (analytic rank), leading coefficient
    lf = solve.l_function(coeffs, tensors.CONDUCTORS, n_max=3000)
    omega = equation.real_period(coeffs)
    print(f"{'curve':>9} {'rank':>4} {'slope':>6} {'ε':>3} {'ord L':>5} {'L^(r)(1)/r!':>12} {'Ω':>9}")
    for c, name in enumerate(names):
        print(
            f"{name:>9} {tensors.RANKS[c]:>4} {slope[c].item():6.2f} {lf['root_number'][c].item():+3d} "
            f"{lf['order'][c].item():>5} {lf['taylor'][c].item():12.6f} {omega[c].item():9.5f}"
        )
    print(f"11a1: L(E,1)/Ω = {lf['leading'][0].item() / omega[0].item():.10f}  (BSD: #Sha·c_11/#tors² = 1·5/5² = 1/5)")

    # Sato–Tate: the angles a_p = 2√p cos θ_p are distributed like (2/π) sin²θ
    x = (a_p.double() / (2 * primes.double().sqrt())).cpu()  # (C, P) in [−1, 1]
    moments = [(x**k).mean().item() for k in (2, 4)]
    print(f"Sato–Tate moments of a_p/2√p: E[x²] = {moments[0]:.3f} (1/4), E[x⁴] = {moments[1]:.3f} (1/8)")

    frames = torch.logspace(math.log10(60), math.log10(x_max), 45).tolist()
    return plot.animate(
        names, tensors.RANKS, primes.cpu(), x, log_prod.cpu(), lf, frames, out_dir / "birch_swinnerton_dyer.gif"
    )


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/maths/birch_swinnerton_dyer/equation.py
r"""Mathematics → tensor translation.

1. Point counting. For odd p, completing the square turns y² + a1xy + a3y = x³ + a2x² + a4x + a6 into
   Y² = f(x) = 4x³ + b2x² + 2b4x + b6 (Y = 2y + a1x + a3), so #E(F_p) = p + 1 + Σ_x χ_p(f(x)) with the Legendre symbol
   χ_p(v) = 1 (nonzero square), −1 (non-square), 0 (v ≡ 0). Hence the Frobenius trace

       a_p = p + 1 − #E(F_p) = −Σ_{x ∈ F_p} χ_p(f(x)),

   evaluated for a whole block of primes and every x at once: χ from a table of squares y² mod p (scatter).

2. The L-function L(E, s) = Σ a_n n^{−s} (a_n multiplicative, a_{p^k} = a_p a_{p^{k−1}} − p a_{p^{k−2}} for good p)
   satisfies Λ(s) = (√N/2π)^s Γ(s) L(E, s) = ε Λ(2 − s) (modularity: Wiles, Taylor–Wiles, BCDT). With
   F(t) = Σ_n a_n e^{−2πnt/√N}:

       Λ(s) = ∫_1^∞ F(t) (t^{s−1} + ε t^{1−s}) dt,    L^{(r)}(E, 1) = (2π/√N) · 2 ∫_1^∞ F(t) (ln t)^r dt  if ε = (−1)^r

   and the integrals become an einsum over a (n, u) grid with t = e^u.
"""

import math

import numpy as np
import torch


def b_invariants(coeffs: torch.Tensor) -> tuple[torch.Tensor, ...]:
    """b2, b4, b6, b8 of each curve.   (C, 5) → four (C,) tensors"""
    a1, a2, a3, a4, a6 = coeffs.unbind(-1)
    b2 = a1 * a1 + 4 * a2
    b4 = 2 * a4 + a1 * a3
    b6 = a3 * a3 + 4 * a6
    b8 = a1 * a1 * a6 + 4 * a2 * a6 - a1 * a3 * a4 + a2 * a3 * a3 - a4 * a4
    return b2, b4, b6, b8


def discriminant(coeffs: torch.Tensor) -> torch.Tensor:
    """Δ = −b2²b8 − 8b4³ − 27b6² + 9b2b4b6; the curve is singular mod p exactly when p | Δ.   (C,)"""
    b2, b4, b6, b8 = b_invariants(coeffs)
    return -b2 * b2 * b8 - 8 * b4**3 - 27 * b6 * b6 + 9 * b2 * b4 * b6


def frobenius_traces(coeffs: torch.Tensor, primes: torch.Tensor, block: int = 3_000_000) -> torch.Tensor:
    r"""a_p = −Σ_x χ_p(f(x)) for every curve and prime (bad primes included: then a_p ∈ {0, ±1}).
    coeffs: (C, 5), primes: (P,) → (C, P) int64. Primes are processed in blocks of about `block` (p, x) pairs."""
    b2, b4, b6, _ = (b[:, None, None] for b in b_invariants(coeffs))  # (C, 1, 1)
    out, start = [], 0
    while start < len(primes):
        stop = start + 1
        while stop < len(primes) and (stop + 1 - start) * int(primes[stop]) <= block:
            stop += 1
        p = primes[start:stop]  # (Pb,)
        width = int(p.max())
        x = torch.arange(width, device=p.device)  # (X,)
        inside = x[None, :] < p[:, None]  # (Pb, X): only x ∈ F_p
        pm = p[:, None]
        is_square = torch.zeros(len(p), width, dtype=torch.bool, device=p.device)  # (Pb, X) squares mod p
        is_square.scatter_(1, torch.where(inside, x * x % pm, 0), True)
        xr = x[None, None, :] % pm[None]  # (1, Pb, X)
        f = (((4 * xr + b2) % pm * xr + 2 * b4) % pm * xr + b6) % pm  # Horner: 4x³ + b2x² + 2b4x + b6 mod p
        chi = torch.where(torch.gather(is_square.expand(len(coeffs), -1, -1), 2, f), 1, -1)  # (C, Pb, X)
        chi = torch.where((f == 0) | ~inside[None], 0, chi)
        a_p = -chi.sum(dim=-1)
        two = p == 2  # completing the square needs 2 to be invertible: count F_2 points directly
        if two.any():
            a_p[:, two] = (2 - _affine_count_mod2(coeffs))[:, None]
        out.append(a_p)
        start = stop
    return torch.cat(out, dim=1)


def _affine_count_mod2(coeffs: torch.Tensor) -> torch.Tensor:
    """Number of (x, y) ∈ F_2² on each curve, by checking all four pairs.   (C,)"""
    a1, a2, a3, a4, a6 = (c[:, None] for c in coeffs.unbind(-1))
    x = torch.tensor([0, 0, 1, 1], device=coeffs.device)
    y = torch.tensor([0, 1, 0, 1], device=coeffs.device)
    lhs = y * y + a1 * x * y + a3 * y - (x**3 + a2 * x * x + a4 * x + a6)
    return (lhs % 2 == 0).sum(dim=1)


def real_period(coeffs: torch.Tensor) -> torch.Tensor:
    r"""Ω = ∫_{E(R)} |dx / (2y + a1x + a3)| = c_∞ · 2∫_{e1}^∞ dx/√f(x), with e1 the largest real root of
    f = 4x³ + b2x² + 2b4x + b6 and c_∞ = 2 when E(R) has two components (Δ > 0). Substituting x = e1 + tan²φ
    removes the endpoint singularity.   (C, 5) → (C,)"""
    b2, b4, b6, _ = (b.double() for b in b_invariants(coeffs))
    companion = torch.zeros(len(coeffs), 3, 3, dtype=torch.float64, device=coeffs.device)
    companion[:, 0] = torch.stack([-b2 / 4, -2 * b4 / 4, -b6 / 4], dim=1)
    companion[:, 1, 0] = companion[:, 2, 1] = 1.0
    roots = torch.linalg.eigvals(companion)  # (C, 3) roots of f/4
    e1 = torch.where(roots.imag.abs() < 1e-9, roots.real, -math.inf).max(dim=1).values  # largest real root
    nodes, weights = (torch.tensor(v, device=coeffs.device) for v in np.polynomial.legendre.leggauss(200))
    phi = math.pi / 4 * (nodes + 1)  # Gauss–Legendre nodes on (0, π/2): (Φ,)
    w = torch.tan(phi)
    x = e1[:, None] + w * w  # (C, Φ)
    f = ((4 * x + b2[:, None]) * x + 2 * b4[:, None]) * x + b6[:, None]
    integrand = 2 * w / torch.cos(phi) ** 2 / torch.sqrt(f)  # dx = 2w sec²φ dφ
    half = math.pi / 4 * integrand @ weights  # ∫_{e1}^∞ dx/√f; the integrand is smooth on [0, π/2]
    components = torch.where(discriminant(coeffs) > 0, 2.0, 1.0)
    return components * 2 * half


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/maths/birch_swinnerton_dyer/tensors.py
"""Build the concrete tensors: five reference elliptic curves of rank 0 … 4 and the primes."""

import math

import torch

# Smallest-conductor curves of each rank (Cremona / LMFDB labels), y² + a1·xy + a3·y = x³ + a2·x² + a4·x + a6.
CURVES = {
    "11a1": (0, -1, 1, -10, -20),
    "37a1": (0, 0, 1, -1, 0),
    "389a1": (0, 1, 1, -2, 0),
    "5077a1": (0, 0, 1, -7, 6),
    "234446a1": (1, -1, 0, -79, 289),
}
RANKS = [0, 1, 2, 3, 4]  # Mordell–Weil ranks over Q (known, e.g. from explicit generators and descent)
CONDUCTORS = [11, 37, 389, 5077, 234446]


def curve_coefficients(device: torch.device) -> torch.Tensor:
    """[a1, a2, a3, a4, a6] for every curve: (C, 5) int64."""
    return torch.tensor(list(CURVES.values()), dtype=torch.int64, device=device)


def primes_up_to(x_max: int, device: torch.device) -> torch.Tensor:
    """All primes p ≤ x_max (sieve of Eratosthenes): (P,) int64."""
    is_prime = torch.ones(x_max + 1, dtype=torch.bool, device=device)
    is_prime[:2] = False
    for p in range(2, math.isqrt(x_max) + 1):
        if is_prime[p]:
            is_prime[p * p :: p] = False
    return torch.nonzero(is_prime).flatten()


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/maths/birch_swinnerton_dyer/solve.py
"""Frobenius traces → partial products and a rank estimate; Dirichlet coefficients → L(E, s) and its order at s = 1."""

import math

import numpy as np
import torch

from .equation import frobenius_traces
from .tensors import primes_up_to


def log_partial_products(coeffs: torch.Tensor, primes: torch.Tensor) -> torch.Tensor:
    """log ∏_{p ≤ X} N_p/p for X running over the primes, good primes only (N_p = p + 1 − a_p).   → (C, P)"""
    from .equation import discriminant

    a_p = frobenius_traces(coeffs, primes)  # (C, P)
    good = discriminant(coeffs)[:, None] % primes[None, :] != 0  # (C, P)
    p = primes.double()[None, :]
    terms = torch.where(good, torch.log((p + 1 - a_p) / p), 0.0)
    return torch.cumsum(terms, dim=1)


def rank_estimate(log_prod: torch.Tensor, primes: torch.Tensor, start: float = 0.5) -> torch.Tensor:
    """BSD heuristic ∏_{p ≤ X} N_p/p ≈ C (log X)^r: least-squares slope of log ∏ against log log X over
    X ∈ [X_max^start, X_max].   (C, P) → (C,)"""
    keep = primes >= primes[-1].double() ** start
    u = torch.log(torch.log(primes[keep].double()))  # (K,)
    v = log_prod[:, keep]  # (C, K)
    uc = u - u.mean()
    return torch.einsum("ck,k->c", v - v.mean(dim=1, keepdim=True), uc) / (uc @ uc)


def dirichlet_coefficients(coeffs: torch.Tensor, n_max: int) -> torch.Tensor:
    """a_n for n = 0 … n_max (a_0 unused) of every curve, from a_p by multiplicativity: start from a = δ_{n,1} and
    multiply in the local factor of each prime, Σ_k a_{p^k} p^{−ks}, with strided slices.   → (C, n_max + 1)"""
    from .equation import discriminant

    primes = primes_up_to(n_max, coeffs.device)
    a_p = frobenius_traces(coeffs, primes).double()  # (C, P)
    bad = discriminant(coeffs)[:, None] % primes[None, :] == 0
    a = torch.zeros(len(coeffs), n_max + 1, dtype=torch.float64, device=coeffs.device)
    a[:, 1] = 1.0
    for i, p in enumerate(primes.tolist()):
        local = [torch.ones_like(a_p[:, i]), a_p[:, i]]  # a_{p^0}, a_{p^1}
        while p ** len(local) <= n_max:  # a_{p^k} = a_p a_{p^{k−1}} − p a_{p^{k−2}}  (bad p: a_p^k)
            nxt = a_p[:, i] * local[-1] - torch.where(bad[:, i], 0.0, p * local[-2])
            local.append(nxt)
        new = a.clone()
        for k in range(1, len(local)):
            q = p**k
            new[:, q::q] += local[k][:, None] * a[:, 1 : n_max // q + 1]  # a_{q m} gets a_q a_m (p ∤ m so far)
        a = new
    return a


def l_function(coeffs: torch.Tensor, conductors: list[int], n_max: int, max_order: int = 4) -> dict:
    """Root number ε, the derivatives L^{(r)}(E, 1) for r ≡ parity of ε, the analytic rank and L(E, s) on a real grid.

    Λ(s) = ∫_1^∞ F(t)(t^{s−1} + ε t^{1−s}) dt with F(t) = Σ_n a_n e^{−2πnt/√N}; t = e^u turns every integral into
    a weighted sum over Gauss–Legendre nodes u_m: G_{nm} = e^{−2πn e^{u_m}/√N}, and Σ_n a_n G_{nm} is an einsum.
    """
    a = dirichlet_coefficients(coeffs, n_max)  # (C, n_max + 1)
    n = torch.arange(1, n_max + 1, dtype=torch.float64, device=coeffs.device)
    s_grid = torch.linspace(0.5, 1.5, 101, dtype=torch.float64, device=coeffs.device)
    nodes, weights = np.polynomial.legendre.leggauss(1500)
    m = torch.tensor((nodes + 1) / 2, device=coeffs.device)  # Gauss–Legendre nodes on (0, 1)
    gl_w = torch.tensor(weights / 2, device=coeffs.device)
    eps, derivs, curves = [], [], []
    for c, big_n in enumerate(conductors):
        scale = 2 * math.pi / math.sqrt(big_n)
        u_max = math.log(45 / scale)  # e^{−scale·e^u} < e^{−45} beyond
        u = u_max * m**2  # dense near u = 0, where the large-n terms live
        w = gl_w * 2 * u_max * m  # Gauss–Legendre weights in m times du/dm
        kernel = torch.exp(-scale * n[:, None] * torch.exp(u)[None, :])  # (n, U)
        f_u = torch.einsum("n,nu->u", a[c, 1:], kernel)  # F(e^u)
        # root number from the functional equation F(1/y) = ε y² F(y), at y = 1.2
        y = torch.tensor([1.2, 1 / 1.2], dtype=torch.float64, device=coeffs.device)
        f_y = torch.einsum("n,ny->y", a[c, 1:], torch.exp(-scale * n[:, None] * y[None, :]))
        e = int(torch.round(f_y[1] / (y[0] ** 2 * f_y[0])).item())
        weight = f_u * torch.exp(u) * w  # F(t) dt with t = e^u
        powers = torch.stack([u**r for r in range(max_order + 1)])  # (R, U): (ln t)^r
        d = 2 * scale * torch.einsum("ru,u->r", powers, weight)  # L^{(r)}(1), valid when ε = (−1)^r
        parity = torch.tensor([(-1) ** r == e for r in range(max_order + 1)], device=coeffs.device)
        derivs.append(torch.where(parity, d, 0.0))
        lam = torch.einsum(
            "su,u->s", torch.exp(torch.outer(s_grid - 1, u)) + e * torch.exp(torch.outer(1 - s_grid, u)), weight
        )
        curves.append(lam * scale**s_grid / torch.exp(torch.lgamma(s_grid)))  # L(s) = Λ(s) (2π/√N)^s / Γ(s)
        eps.append(e)
    derivs = torch.stack(derivs)  # (C, R)
    order = torch.argmax((derivs.abs() > 1e-6).int(), dim=1)  # first derivative that does not vanish
    fact = torch.tensor([math.factorial(r) for r in range(max_order + 1)], dtype=torch.float64, device=coeffs.device)
    leading = derivs.gather(1, order[:, None])[:, 0]
    return {
        "root_number": torch.tensor(eps),
        "derivatives": derivs,
        "order": order,
        "leading": leading,
        "taylor": leading / fact[order],
        "s": s_grid,
        "L": torch.stack(curves),
    }


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/maths/birch_swinnerton_dyer/plot.py
"""Partial products against log log X, a_p inside the Hasse band, the Sato–Tate histogram and L(E, s) near s = 1."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from problems._common.plotting import ACCENT, frame_indices, save_gif


def _slope(loglog: np.ndarray, v: np.ndarray) -> float:
    uc = loglog - loglog.mean()
    return float(((v - v.mean()) * uc).sum() / (uc * uc).sum())


def animate(names, ranks, primes, x, log_prod, lf, frames, path: Path) -> Path:
    p = primes.numpy().astype(float)
    x, log_prod = x.numpy(), log_prod.numpy()
    loglog = np.log(np.log(p))
    fig, ((ax_g, ax_a), (ax_h, ax_l)) = plt.subplots(2, 2, figsize=(10, 6.6))

    lines = [
        ax_g.plot([], [], color=ACCENT[c], lw=1.4, label=f"{name}, rank {r}")[0]
        for c, (name, r) in enumerate(zip(names, ranks))
    ]
    ax_g.set_xlim(loglog[0], loglog[-1] + 0.05)
    ax_g.set_ylim(min(0.0, log_prod.min()) - 0.3, log_prod.max() + 0.5)
    ax_g.set_xlabel(r"$\log\log X$")
    ax_g.set_ylabel(r"$\log \prod_{p \leq X} N_p / p$")
    legend = ax_g.legend(fontsize=7, loc="upper left")
    title_g = ax_g.set_title("", fontsize=9)

    pick = 1  # 37a1
    dots = ax_a.scatter([], [], s=2, color=ACCENT[pick])
    for level in (-1, 1):
        ax_a.axhline(level, color="#dc2626", lw=1, ls="--")
    ax_a.set_xlim(0, p[-1])
    ax_a.set_ylim(-1.15, 1.15)
    ax_a.set_xlabel("$p$")
    ax_a.set_ylabel(r"$a_p / 2\sqrt{p}$")
    ax_a.set_title(f"{names[pick]}: $N_p = p + 1 - a_p$ inside the Hasse bound", fontsize=9)

    edges = np.linspace(-1, 1, 25)
    hist = ax_h.stairs(np.zeros(len(edges) - 1), edges, fill=True, color="#93c5fd", label="all five curves")
    grid = np.linspace(-1, 1, 200)
    ax_h.plot(
        grid, 2 / np.pi * np.sqrt(1 - grid**2), color=ACCENT[1], lw=1.5, label=r"Sato–Tate $\frac{2}{\pi}\sqrt{1-x^2}$"
    )
    ax_h.set_xlim(-1, 1)
    ax_h.set_ylim(0, 1.0)
    ax_h.set_xlabel(r"$x = a_p / 2\sqrt{p}$")
    ax_h.legend(fontsize=7, loc="lower center")
    title_h = ax_h.set_title("", fontsize=9)

    s, curve = lf["s"].cpu().numpy(), lf["L"].cpu().numpy()
    for c, name in enumerate(names):
        ax_l.plot(s, curve[c], color=ACCENT[c], lw=1.4, label=f"{name}: order {int(lf['order'][c])}")
    ax_l.axhline(0, color="#9ca3af", lw=0.8)
    ax_l.axvline(1, color="#9ca3af", lw=0.8, ls="--")
    ax_l.set_ylim(-0.6, 1.6)
    ax_l.set_xlabel("$s$")
    ax_l.set_ylabel("$L(E, s)$")
    ax_l.legend(fontsize=7, loc="upper left")
    ax_l.set_title("$L(E, s)$ vanishes at $s = 1$ to order = rank", fontsize=9)

    def update(f: int):
        top = frames[f]
        k = int(np.searchsorted(p, top, side="right"))
        window = (p >= top**0.5) & (p <= top)
        for c, line in enumerate(lines):
            line.set_data(loglog[:k], log_prod[c, :k])
            if window.sum() > 3:
                est = _slope(loglog[window], log_prod[c, window])
                legend.get_texts()[c].set_text(f"{names[c]}, rank {ranks[c]}: slope {est:.2f}")
        title_g.set_text(f"growth like $(\\log X)^r$ — primes up to X = {int(top):,}")
        dots.set_offsets(np.c_[p[:k], x[pick, :k]])
        hist.set_data(np.histogram(x[:, :k].ravel(), bins=edges, density=True)[0])
        title_h.set_text(f"Sato–Tate distribution ({5 * k:,} values)")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(frames), len(frames)), path, fps=6)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'x_max': 30000
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.maths.birch_swinnerton_dyer")]:
    del sys.modules[name]
main = importlib.import_module("problems.maths.birch_swinnerton_dyer.main")
problem_dir = Path("problems/maths/birch_swinnerton_dyer")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
